# Melanoma PDX reproduction

This notebook trains Xenocomm on the reduced melanoma PDX dataset and recreates the core ligand, receptor, and target analyses.

PDX expression is rebuilt from raw counts in `.raw.X`: normalize each cell to 10,000 counts, then apply natural-log1p. The original counts are retained in `layers["counts"]`; saved expression and gene-selection statistics are not used.


In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import xenocomm as xc
from scipy import sparse

sns.set_theme(context="notebook", style="whitegrid")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
DATA_DIR = NOTEBOOK_DIR / "data/melanoma_pdx_10k"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs/pdx_10k"
MOUSE_H5AD = DATA_DIR / "adata_mouse.h5ad"
HUMAN_H5AD = DATA_DIR / "adata_human.h5ad"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def read_pdx_counts(path):
    stored = ad.read_h5ad(path)
    if stored.raw is None:
        raise ValueError(f"{path} must contain raw counts in .raw.X")
    counts = sparse.csr_matrix(stored.raw.X, copy=True)
    if (
        not np.isfinite(counts.data).all()
        or np.any(counts.data < 0)
        or np.any(counts.data != np.floor(counts.data))
    ):
        raise ValueError(f"{path}: .raw.X must contain nonnegative integer counts")
    if np.any(np.asarray(counts.sum(axis=1)).ravel() <= 0):
        raise ValueError(f"{path}: every cell must have a positive count total")
    data = ad.AnnData(
        X=counts.astype(np.float32),
        obs=stored.obs[["sample", "cell_type"]].copy(),
        var=stored.raw.var[["gene_id"]].copy(),
    )
    data.layers["counts"] = counts
    sc.pp.normalize_total(data, target_sum=10_000)
    sc.pp.log1p(data)
    return data


adata_mouse = read_pdx_counts(MOUSE_H5AD)
adata_human = read_pdx_counts(HUMAN_H5AD)
print(f"Mouse: {adata_mouse.n_obs:,} cells; human: {adata_human.n_obs:,} cells")


In [ ]:
STEPS_PER_BATCH = 250
EPOCHS = 5
POSTERIOR_SAMPLES = 2_000
TRAINING_SEED = 20260716
POSTERIOR_SEED = 20260717
VALIDATION_CELLS = 4096
VALIDATION_SPLIT_SEED = 20260717
VALIDATION_SEED = 20260718


def validation_indices(obs, size, seed):
    strata = obs[["sample", "cell_type"]].astype(str).agg("|".join, axis=1)
    counts = strata.value_counts().sort_index()
    exact = counts.to_numpy() * size / len(obs)
    quotas = np.floor(exact).astype(int)
    quotas[np.argsort(-(exact - quotas), kind="stable")[: size - quotas.sum()]] += 1
    rng = np.random.RandomState(seed)
    held_out = np.concatenate([
        rng.choice(np.flatnonzero(strata.to_numpy() == label), quota, replace=False)
        for label, quota in zip(counts.index, quotas, strict=True)
    ])
    rng.shuffle(held_out)
    training = np.setdiff1d(np.arange(len(obs)), held_out)
    return training, held_out


In [ ]:
network = xc.prepare_network(adata_mouse, adata_human, dispersion_cutoff=-10)
training_indices, held_out_indices = validation_indices(
    adata_mouse.obs, VALIDATION_CELLS, VALIDATION_SPLIT_SEED
)
ligand_abundance = xc.compute_ligand_abundance(
    adata_mouse,
    adata_human,
    network["ligands"],
    network["human_ligands"],
    network["ligand_receptor_matrix"],
)
model = xc.XenocommModel(
    adata_mouse[training_indices],
    **network,
    mean_ligand=ligand_abundance,
    receptor_target_mode="learned",
    training_seed=TRAINING_SEED,
    posterior_seed=POSTERIOR_SEED,
    batch_size=1024,
    steps_per_batch=STEPS_PER_BATCH,
    epochs=EPOCHS,
)
training = model.train(
    validation_mouse=adata_mouse[held_out_indices],
    absolute_tolerance=0.001 * 1024,
    patience=3,
    min_evaluations=5,
    validation_seed=VALIDATION_SEED,
)
training


In [ ]:
samples = model.sample(POSTERIOR_SAMPLES)
parameters = model.get_parameters()
ligand_results = xc.ligand_result_table(
    model.ligands,
    samples,
    model.mean_ligand_np,
    model.ligand_receptor_matrix_np,
    xc.get_receptor_sensitivity(parameters),
)

payload = {
    "ligands": np.asarray(model.ligands),
    "human_ligands": np.asarray(model.human_ligands),
    "receptors": np.asarray(model.receptors),
    "targets": np.asarray(model.targets),
    "mean_ligand_np": model.mean_ligand_np,
    "ligand_receptor_matrix_np": model.ligand_receptor_matrix_np,
    **{f"s_{key}": value for key, value in samples.items()},
    **{f"v_{key}": value for key, value in parameters.items()},
}
np.savez(OUTPUT_DIR / "model.staged.npz", **payload)
ligand_results.to_parquet(OUTPUT_DIR / "ligand_results.parquet", index=False)

detected = ligand_results.loc[ligand_results["called"]].head(15).copy()
print(f"Detected ligands: {len(ligand_results.loc[ligand_results['called']])}")
display(detected[["ligand", "delta_h_mean", "human_fraction_mean"]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for data, ax, title in (
    (adata_mouse, axes[0], "Mouse cell types"),
    (adata_human, axes[1], "Human cell types"),
):
    variable_genes = sc.pp.highly_variable_genes(data, n_top_genes=2000, inplace=False)
    embedding = data[:, variable_genes["highly_variable"].to_numpy()].copy()
    sc.pp.pca(embedding, n_comps=30, random_state=0)
    sc.pp.neighbors(embedding, n_neighbors=15, n_pcs=30, random_state=0)
    sc.tl.umap(embedding, random_state=0)
    sc.pl.umap(embedding, color="cell_type", ax=ax, show=False, title=title)
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=detected, x="delta_h_mean", y="ligand", ax=ax, color="#4b9cbe")
ax.set(xlabel="Human counterfactual activation", ylabel="Ligand", title="Top detected ligands")
plt.tight_layout()


In [ ]:
binding = xc.species_bias_df(model, samples, parameters)
binding = binding[binding["ligand"].isin(detected["ligand"])].melt(
    id_vars="ligand",
    value_vars=["mouse_binding", "human_binding"],
    var_name="species",
    value_name="binding",
)
binding["species"] = binding["species"].str.replace("_binding", "", regex=False).str.title()
binding["ligand"] = pd.Categorical(binding["ligand"], detected["ligand"], ordered=True)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=binding, x="binding", y="ligand", hue="species", ax=ax)
ax.set(xlabel="Binding score", ylabel="Ligand", title="Species-decomposed ligand binding")
plt.tight_layout()


In [ ]:
receptors = xc.receptor_marginal_df(model, samples, parameters).nlargest(12, "delta")
receptors_long = receptors.melt(
    id_vars="receptor", value_vars=["mouse", "delta"], var_name="component", value_name="activation"
)
receptors_long["component"] = receptors_long["component"].map(
    {"mouse": "Mouse baseline", "delta": "Human delta"}
)
receptors_long["receptor"] = pd.Categorical(
    receptors_long["receptor"], receptors["receptor"], ordered=True
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=receptors_long, x="activation", y="receptor", hue="component", ax=ax)
ax.set(xlabel="Activation", ylabel="Receptor", title="Top receptor marginal activation")
plt.tight_layout()


In [ ]:
top_ligand = detected.iloc[0]["ligand"]
counterfactual = xc.receptor_counterfactual_df(
    model, samples, parameters, remove_ligands=[top_ligand], label=top_ligand
)
pivot = counterfactual.pivot(index="receptor", columns="component", values="activation").fillna(0)
pivot = pivot.loc[pivot.sum(axis=1).nlargest(12).index]

fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot.barh(stacked=True, ax=ax)
ax.invert_yaxis()
ax.set(xlabel="Receptor activation", ylabel="Receptor", title=f"Counterfactual receptor activation: {top_ligand}")
plt.tight_layout()


In [ ]:
targets = xc.targets_marginal_df(model, samples, parameters, top_n=15)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=targets, x="value", y="target", ax=ax, color="#7a9a01")
ax.set(xlabel="Weighted activation", ylabel="Target", title="Top downstream targets")
plt.tight_layout()
targets


In [ ]:
dot_data = xc.species_dotplot_data(
    model, adata_mouse, adata_human, ligands=detected["ligand"].head(8)
)
dot = pd.DataFrame(dot_data)
fig, ax = plt.subplots(figsize=(7, 3.5))
points = ax.scatter(dot["genes"], dot["species"], s=np.asarray(dot["size"]) * 500, c=dot["color"], cmap="viridis")
ax.set(xlabel="Ligand", ylabel="Species", title="Cross-species ligand expression")
plt.xticks(rotation=45, ha="right")
plt.colorbar(points, ax=ax, label="Mean expression")
plt.tight_layout()
